# Van der Vusse CSTR case study (deterministic variant, up to aggregated causal graph)

This notebook replicates the workflow style from `tutorials/case_studies/biogeoscience_case_study.ipynb`, but uses a **continuous-time** van der Vusse CSTR model.

Even though the model is given by differential equations, causal discovery is still possible by:

1. Simulating the ODE system,
2. Sampling trajectories at regular time intervals,
3. Applying lag-based causal discovery (PCMCIplus) to the sampled multivariate time series.

This is a **separate file** variant with deterministic forcing/noise settings and constant kinetic rates for stability and reproducibility.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

import tigramite.data_processing as pp
import tigramite.plotting as tp
from tigramite.pcmci import PCMCI
from tigramite.independence_tests.robust_parcorr import RobustParCorr

## Data generation and plotting

We simulate the following van der Vusse CSTR states:

- $c_A$: concentration of species A,
- $c_B$: concentration of species B,
- $artheta$: reactor temperature,
- $artheta_K$: coolant temperature.

Additionally we include the feed temperature $artheta_0(t)$ as the **only time-varying external input**.
For robustness in this tutorial variant, kinetic rates are kept constant (evaluated at nominal temperature), and no process/measurement noise is added.


In [ ]:
# ---------- Model parameters (requested nominal values) ----------
params = {
    # kinetic pre-exponentials
    "k10": 1.287e12,      # h^-1
    "k20": 1.287e12,      # h^-1
    "k30": 9.043e9,       # (mol_A * h)^-1
    # activation terms (Arrhenius exponent numerator, in K)
    "E1": -9758.3,
    "E2": -9758.3,
    "E3": -8560.0,
    # reaction enthalpies (kJ/mol)
    "dHrAB": 4.20,
    "dHrBC": -11.0,
    "dHrAD": -41.85,
    # thermo-physical and geometric parameters
    "rho": 0.9342,        # kg/L
    "Cp": 3.01,           # kJ/(kg*K)
    "kw": 4032.0,         # kJ/(h*m^2*K)
    "AR": 0.215,          # m^2
    "VR": 0.01,           # m^3
    "mK": 5.0,            # kg
    "CPK": 2.0,           # kJ/(kg*K)
    # operating values (used for simulation)
    "F": 0.01419,         # m^3/h
    "cA0": 5.1,           # mol/L
    "theta0_nom": 387.34, # K
    "Qk_nom": -1113.5,    # kJ/h
}

# Use constant kinetic rates (evaluated at nominal feed temperature)
def _arrhenius_rates(theta, p):
    k1 = p["k10"] * np.exp(p["E1"] / theta)
    k2 = p["k20"] * np.exp(p["E2"] / theta)
    k3 = p["k30"] * np.exp(p["E3"] / theta)
    return k1, k2, k3

params["k1_const"], params["k2_const"], params["k3_const"] = _arrhenius_rates(params["theta0_nom"], params)

def theta0_profile(t, p):
    # Deterministic exogenous feed-temperature disturbance (no noise)
    periodic = 1.5 * np.sin(2 * np.pi * t / 180.0) + 0.8 * np.sin(2 * np.pi * t / 420.0 + 0.7)
    return p["theta0_nom"] + periodic

def qk_profile(t, p):
    # Keep coolant heat flow fixed at nominal value for this tutorial
    return p["Qk_nom"]

def cstr_rhs(t, x, p):
    cA, cB, theta, thetaK = x

    theta0 = theta0_profile(t, p)
    Qk = qk_profile(t, p)
    k1, k2, k3 = p["k1_const"], p["k2_const"], p["k3_const"]

    F_over_VR = p["F"] / p["VR"]
    rhoCp = p["rho"] * p["Cp"]
    rhoCpVR = rhoCp * (1000.0 * p["VR"])  # VR converted to L

    dcA = F_over_VR * (p["cA0"] - cA) - k1 * cA - k3 * cA**2
    dcB = -F_over_VR * cB + k1 * cA - k2 * cB

    reaction_heat = k1 * cA * p["dHrAB"] + k2 * cB * p["dHrBC"] + k3 * cA**2 * p["dHrAD"]

    dtheta = (
        F_over_VR * (theta0 - theta)
        - reaction_heat / rhoCp
        + (p["kw"] * p["AR"] / rhoCpVR) * (thetaK - theta)
    )

    dthetaK = (Qk + p["kw"] * p["AR"] * (theta - thetaK)) / (p["mK"] * p["CPK"])

    return np.array([dcA, dcB, dtheta, dthetaK])


In [ ]:
# ---------- Generate multiple trajectories ----------
T = 1200
dt = 1.0
t_eval = np.arange(0, T, dt)

M = 1  # deterministic setup: one run is sufficient

var_names = ["cA", "cB", "theta", "thetaK", "theta0"]
N = len(var_names)

data_dict = {}

for m in range(M):
    # Fixed initial conditions (no random perturbations)
    x0 = np.array([2.14, 1.09, 387.34, 386.06])

    sol = solve_ivp(
        lambda t, x: cstr_rhs(t, x, params),
        (t_eval[0], t_eval[-1]),
        x0,
        t_eval=t_eval,
        method="Radau",
        rtol=1e-7,
        atol=1e-9,
        max_step=1.0,
    )

    cA, cB, theta, thetaK = sol.y

    # Deterministic exogenous input sampled on the analysis grid
    theta0 = np.array([theta0_profile(t, params) for t in t_eval])

    data_dict[m] = np.column_stack([cA, cB, theta, thetaK, theta0])

# Create Tigramite DataFrame in multiple-run mode
dataframe = pp.DataFrame(data=data_dict, analysis_mode="multiple", var_names=var_names)


In [ ]:
# Quick look at trajectories
tp.plot_timeseries(
    dataframe,
    figsize=(9, 6),
    grey_masked_samples=False,
    data_linewidth=1.0,
)
plt.show()

## Causal discovery analysis

We now apply sliding-window PCMCIplus to the sampled CSTR trajectories.

### Notes on assumptions

- **Differential equations are fine**: after sampling, we obtain a discrete-time multivariate process suitable for lagged causal discovery.
- We include $\vartheta_0$ as an observed exogenous variable.
- To reflect exogeneity, we forbid incoming links into $\vartheta_0$.

In [ ]:
# Causal discovery on all variables, after removing the startup transient
transient_cut = 120
analysis_data_dict = {m: data_dict[m][transient_cut:, :] for m in data_dict}

dataframe_analysis = pp.DataFrame(
    data=analysis_data_dict,
    analysis_mode="multiple",
    var_names=var_names,
)

# Lagged-only discovery helps directional identifiability in this deterministic setup
tau_min = 1
tau_max = 2

# Independence test
cond_ind_test = RobustParCorr(verbosity=0)

pc_alpha = 0.02
window_step = 120
window_length = 360

# Keep theta0 exogenous: forbid incoming links to theta0
theta0_idx = var_names.index("theta0")
link_assumptions_absent_link_means_no_knowledge = {
    theta0_idx: {
        (i, -tau): ""
        for i in range(N)
        for tau in range(tau_min, tau_max + 1)
    }
}

link_assumptions = PCMCI.build_link_assumptions(
    link_assumptions_absent_link_means_no_knowledge=link_assumptions_absent_link_means_no_knowledge,
    n_component_time_series=N,
    tau_max=tau_max,
    tau_min=tau_min,
)



In [ ]:
pcmci = PCMCI(dataframe=dataframe_analysis, cond_ind_test=cond_ind_test, verbosity=0)

method_args = {
    "tau_min": tau_min,
    "tau_max": tau_max,
    "pc_alpha": pc_alpha,
    "link_assumptions": link_assumptions,
}

summary_results = pcmci.run_sliding_window_of(
    method="run_pcmciplus",
    method_args=method_args,
    window_step=window_step,
    window_length=window_length,
    conf_lev=0.9,
)



In [ ]:
# Plot window-specific causal graphs
graphs = summary_results["window_results"]["graph"]

n_windows = len(graphs)
ncols = min(4, n_windows)
nrows = int(np.ceil(n_windows / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.0 * nrows))
axes = np.atleast_1d(axes).ravel()

for w, ax in enumerate(axes):
    if w < n_windows:
        tp.plot_graph(
            graph=graphs[w],
            var_names=var_names,
            node_size=0.35,
            node_label_size=9,
            arrow_linewidth=2.5,
            show_colorbar=False,
            fig_ax=(fig, ax),
        )
        ax.set_title(f"Window {w + 1}", fontsize=10)
    else:
        ax.axis("off")

fig.tight_layout()
plt.show()

### Aggregated causal graph

As in the biogeoscience case study, we summarize across windows using the most frequent links and their frequencies.

In [ ]:
window_results = summary_results["summary_results"]
val_key = "val_matrix_mean" if "val_matrix_mean" in window_results else "val_matrix"
val_tmp = np.abs(window_results[val_key])
val_tmp[np.arange(N), np.arange(N), :] = 0.0
cross_max = np.abs(val_tmp).max()

graph = window_results["most_frequent_links"]
link_width = window_results["link_frequency"]

fig, ax = tp.plot_graph(
    graph=graph,
    link_width=link_width,
    cmap_edges="RdBu_r",
    arrow_linewidth=4.0,
    node_size=0.4,
    var_names=var_names,
    vmin_edges=-max(0.2, cross_max),
    vmax_edges=max(0.2, cross_max),
    edge_ticks=0.5,
    node_ticks=0.5,
    show_colorbar=False,
    figsize=(4.5, 3.6),
)
plt.show()

